In [1]:
pip install snowflake-connector-python pandas python-dotenv


  Using cached snowflake_connector_python-3.17.4-cp312-cp312-macosx_11_0_x86_64.whl.metadata (74 kB)
  Using cached boto3-1.40.45-py3-none-any.whl.metadata (6.7 kB)
  Using cached botocore-1.40.45-py3-none-any.whl.metadata (5.7 kB)
  Using cached cryptography-46.0.2-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached pyopenssl-25.3.0-py3-none-any.whl.metadata (17 kB)
INFO: pip is looking at multiple versions of cryptography to determine which version is compatible with other requirements. This could take a while.
  Using cached cryptography-46.0.1-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached cryptography-46.0.0-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached s3transfer-0.14.0-py3-none-any.whl.metadata (1.7 kB)
Using cached snowflake_connector_python-3.17.4-cp312-cp312-macosx_11_0_x86_64.whl (1.0 MB)
Using cached pyopenssl-25.3.0-py3-none-any.whl (57 kB)
Using cached cryptography-46.0.0-cp311-abi3-macosx_10_9_universal2

In [ ]:
import os
from dotenv import load_dotenv
import snowflake.connector
import pandas as pd

# Load environment file 
load_dotenv("file.env")

USER = os.getenv("SNOWFLAKE_USER")
PWD  = os.getenv("SNOWFLAKE_PASSWORD")
WH   = os.getenv("SNOWFLAKE_WAREHOUSE")
DB   = os.getenv("SNOWFLAKE_DATABASE")
SC   = os.getenv("SNOWFLAKE_SCHEMA")
ROLE = os.getenv("SNOWFLAKE_ROLE")

#  Candidate connection options to try
candidates = [
    ("gmrzyvu-ry13975", None),                            # global URL
    ("gmrzyvu-ry13975.europe-west2.gcp", None),           # region+cloud
    ("mj62815.europe-west2.gcp", None),                   # locator+region+cloud
    ("gmrzyvu-ry13975", "gmrzyvu-ry13975.snowflakecomputing.com"),  # force host
]

conn = None

# --- Try to connect using whichever works ---
for acct, host in candidates:
    try:
        print(f"Trying account={acct} host={host}")
        kwargs = dict(
            user=USER, password=PWD, account=acct,
            warehouse=WH, database=DB, schema=SC, role=ROLE
        )
        if host:
            kwargs["host"] = host
        conn = snowflake.connector.connect(**kwargs)
        cur = conn.cursor()
        cur.execute("select current_account(), current_region(), current_version()")
        print(" Connected successfully:", cur.fetchone())
        cur.close()
        break
    except Exception as e:
        print(" Failed:", e)

if conn is None:
    raise ConnectionError("Could not connect to any Snowflake candidate.")

#  Export setup 
OUTPUT_DIR = "finrisk_exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tables = [
    "CUSTOMER_INFO",
    "FRAUDULENT_PATTERNS",
    "MERCHANT_INFO",
    "TRANSACTIONS",
    "TRANSACTIONS_FLAGS",
    "TRANSACTION_PATTERNS"
]

#  Export all tables to CSV files
for table in tables:
    print(f"📦 Exporting {table} ...")
    query = f"SELECT * FROM {DB}.{SC}.{table};"
    df = pd.read_sql(query, conn)
    
    file_path = os.path.join(OUTPUT_DIR, f"{table}.csv")
    df.to_csv(file_path, index=False)
    print(f" Saved {table} → {file_path}")

conn.close()
print("\n All 6 tables exported successfully to the 'finrisk_exports' folder.")


Trying account=gmrzyvu-ry13975 host=None
✅ Connected successfully: ('MJ62815', 'GCP_EUROPE_WEST2', '9.30.0')
📦 Exporting CUSTOMER_INFO ...


/var/folders/mp/37ndhfvn70xdgjmd9pxbyzm80000gn/T/ipykernel_27354/984780272.py:65: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Saved CUSTOMER_INFO → finrisk_exports/CUSTOMER_INFO.csv
📦 Exporting FRAUDULENT_PATTERNS ...
✅ Saved FRAUDULENT_PATTERNS → finrisk_exports/FRAUDULENT_PATTERNS.csv
📦 Exporting MERCHANT_INFO ...
✅ Saved MERCHANT_INFO → finrisk_exports/MERCHANT_INFO.csv
📦 Exporting TRANSACTIONS ...
✅ Saved TRANSACTIONS → finrisk_exports/TRANSACTIONS.csv
📦 Exporting TRANSACTIONS_FLAGS ...
✅ Saved TRANSACTIONS_FLAGS → finrisk_exports/TRANSACTIONS_FLAGS.csv
📦 Exporting TRANSACTION_PATTERNS ...
✅ Saved TRANSACTION_PATTERNS → finrisk_exports/TRANSACTION_PATTERNS.csv

🎯 All 6 tables exported successfully to the 'finrisk_exports' folder.


In [4]:
data = pd.read_csv("finrisk_exports/CUSTOMER_INFO.csv")
data.head()

,CUSTOMER_ID,FIRST_NAME,LAST_NAME,DOB,EMAIL,PHONE_NUMBER,ADDRESS,REGION,CITY,ACCOUNT_STATUS,ACCOUNT_TYPE,DATE_CREATED
0,1,Olivia,Smith,1970-01-01,olivia.smith@email.com,+44 7911 654321,"456 High Street, Manchester, M2 3CD",Greater Manchester,Manchester,Active,Checking,2023-03-05 14:00:00
1,2,Amara,Okafor,1970-01-01,amara.okafor@email.com,+44 7911 987654,"789 Piccadilly, Manchester, M3 4EF",Greater Manchester,Manchester,Suspended,Savings,2023-04-10 11:30:00
2,3,Ivan,Petrov,1970-01-01,ivan.petrov@email.com,+44 7911 654987,"321 Oxford Road, Manchester, M2 4AB",Greater Manchester,Manchester,Active,Credit,2023-02-25 08:00:00
3,4,Marta,Kowalska,1970-01-01,marta.kowalska@email.com,+44 7911 112233,"654 Deansgate, Manchester, M3 2QX",Greater Manchester,Manchester,Active,Savings,2023-04-12 15:00:00
4,5,Kwame,Asamoah,1970-01-01,kwame.asamoah@email.com,+44 7911 223344,"213 King Street, Manchester, M1 1AD",Greater Manchester,Manchester,Active,Checking,2023-01-18 13:45:00


In [5]:
data_2 = pd.read_csv("finrisk_exports/TRANSACTIONS.csv")
data_2.head()

,TRANSACTION_ID,CUSTOMER_ID,MERCHANT_ID,TRANSACTION_DATE,TRANSACTION_TYPE,AMOUNT,TRANSACTION_STATUS,TRANSACTION_ADDRESS,TRANSACTION_CITY,TRANSACTION_REGION,CHANNEL,TRANSACTION_DEVICE,FRAUD_FLAG
0,1,3,7,2023-06-15 09:00:00.000,Debit,150.75,Completed,"123 Market Street, Manchester, M1 2AB",Manchester,Greater Manchester,Online,Mobile,False
1,2,5,12,2023-06-16 14:15:00.000,Credit,120.00,Completed,"45 Oxford Road, Manchester, M2 4BB",Manchester,Greater Manchester,In-Store,ATM,True
2,3,9,6,2023-06-17 10:30:00.000,Debit,500.00,Failed,"78 Deansgate, Manchester, M3 2LQ",Manchester,Greater Manchester,Online,Laptop,False
3,4,15,19,2023-06-18 13:20:00.000,Credit,350.00,Completed,"123 King Street, Manchester, M1 6JQ",Manchester,Greater Manchester,In-Store,POS Terminal,True
4,5,2,20,2023-06-19 16:40:00.000,Debit,200.00,Completed,"654 Deansgate, Manchester, M3 2QX",Manchester,Greater Manchester,Online,Mobile,False


In [6]:
data_3 = pd.read_csv("finrisk_exports/FRAUDULENT_PATTERNS.csv")
data_3.head()

,PATTERN_ID,PATTERN_NAME,PATTERN_DETAILS,SEVERITY_LEVEL,IS_ACTIVE
0,1,Credit Card Fraud,Unauthorized credit card transactions in retai...,High,True
1,2,Location Anomaly,Transactions flagged due to unusual locations ...,Medium,True
2,3,High-Value Purchase,"Purchases over a specified value (e.g., Â£1000...",High,True
3,4,Multiple Transactions in Short Time,"Multiple transactions made in a short period, ...",Medium,True
4,5,Account Takeover,A userâ€™s account is accessed by an unauthori...,High,True
